# Optimized Gradient Method (OGM)

This notebook applies the systematic Lyapunov-function discovery procedure to the 
Optimized Gradient Method, proposed in "Optimized first-order methods for smooth 
convex minimization" of Kim and Fessler (2016). This was later shown to be exact
optimal for reducing the function value suboptimality $f(x_N) - f(x_\star)$ in 
$L$-smooth convex minimization, in "The exact information-based complexity of 
smooth convex minimization" by Drori (2017).

## Import the required libraries

In [1]:
import functools
import itertools

import numpy as np
import pepflow as pf
import pepflow.lyapunov_utils as lu
import sympy as sp
from IPython.display import Math, display

## Define the function class

In [2]:
L = pf.Parameter("L")
f = pf.SmoothConvexFunction(is_basis=True, tags=["f"], L=L)

## Write a function that generates a PEPContext object associated with OGM

In [3]:
@functools.cache
def theta(i: int, N: int):
    if i == -1:
        return 0
    if i == N:
        return 1 / 2 * (1 + np.sqrt(8 * theta(N - 1, N) ** 2 + 1))
    return 1 / 2 * (1 + np.sqrt(4 * theta(i - 1, N) ** 2 + 1))


def make_ctx_ogm(
    ctx_name: str, N: int | sp.Integer, stepsize: pf.Parameter
) -> pf.PEPContext:
    N = int(N)
    ctx_ogm = pf.PEPContext(ctx_name).set_as_current()
    x = pf.Vector(is_basis=True, tags=["x_0"])
    z = x
    f.set_stationary_point("x_star")

    for i in range(N):
        y = x - stepsize * f.grad(x)
        z = z - 2 * stepsize * theta(i, N) * f.grad(x)
        z.add_tag(f"z_{i + 1}")
        x = (1 - 1 / theta(i + 1, N)) * y + 1 / theta(i + 1, N) * z
        x.add_tag(f"x_{i + 1}")

    return ctx_ogm

## Numerical PEP solve 
### Find tight and sparsified proof certificates:

In [4]:
N = 5
N_int = int(N)
R = pf.Parameter("R")
L_value = sp.S(1)
R_value = sp.S(1)

ctx_prf = make_ctx_ogm(ctx_name="ctx_prf", N=N, stepsize=1 / L)
pb_prf = pf.PEPBuilder(ctx_prf)
pb_prf.add_initial_constraint(
    ((ctx_prf["x_0"] - ctx_prf["x_star"]) ** 2).le(R, name="initial_condition")
)
pb_prf.set_performance_metric(f(ctx_prf[f"x_{N_int}"]) - f(ctx_prf["x_star"]))

result_dense = pb_prf.solve(resolve_parameters={"L": L_value, "R": R_value})
lamb_dense = result_dense.get_scalar_constraint_dual_value_in_numpy(f)

print("dense optimum:", result_dense.opt_value)

dense optimum: 0.01858694825943362


In [5]:
pf.pprint_labeled_matrix(lamb_dense.matrix, lamb_dense.row_names, lamb_dense.col_names)

<IPython.core.display.Math object>

### The $\lambda$ certificates are already sparse, but double-check by enforcing relaxed constraints

In [6]:
def tag_to_index(tag: str, N: int | sp.Integer = N) -> int:
    if tag == "x_star":
        return int(N) + 1
    return int(tag.split("_", 1)[1])


def ogm_relaxed_constraints_from_observed_pattern(
    lamb_matrix, N: int | sp.Integer
) -> list[str]:
    relaxed_constraints = []
    N = int(N)
    for tag_i in lamb_matrix.row_names:
        i = tag_to_index(tag_i, N)
        if i == N + 1:
            continue
        for tag_j in lamb_matrix.col_names:
            j = tag_to_index(tag_j, N)
            if i < N and i + 1 == j:
                continue
            relaxed_constraints.append(f"f:{tag_i},{tag_j}")
    return relaxed_constraints


pb_prf.set_relaxed_constraints(
    ogm_relaxed_constraints_from_observed_pattern(lamb_dense, N)
)
result = pb_prf.solve(resolve_parameters={"L": L_value, "R": R_value})

# Dual variable associated with the initial condition.
tau_sol = result.dual_var_manager.dual_value("initial_condition")
# Dual variables associated with the interpolation conditions of f.
lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(f)
# Dual matrix associated with the Gram matrix.
S_sol = result.get_gram_dual_matrix()

print("sparse optimum:", result.opt_value)
print("tau:", tau_sol)

sparse optimum: 0.018588926661423158
tau: 0.018588155914589344


In [7]:
lamb_sol.pprint()

<IPython.core.display.Math object>

In [8]:
S_sol.pprint()

<IPython.core.display.Math object>

### The Gram dual matrix has rank 1

In [9]:
np.linalg.matrix_rank(S_sol.matrix)

np.int64(1)

---

## Step 1. Propose a candidate Lyapunov function

### Let $\mathcal I_0=\{(\star,0)\},\,\, \mathcal I_k=\{(j,j+1):j=0,\ldots,k-1\}\cup\{(\star,j):j=0,\ldots,k\}$ for $k=1,\dots,N$ and $\mathcal J_k = \emptyset$ for $k=0,\dots,N$.

In [10]:
def named_matrix_entry(named_matrix, row_name: str, col_name: str):
    row = named_matrix.row_names.index(row_name)
    col = named_matrix.col_names.index(col_name)
    return named_matrix.matrix[row, col]


def lamb_numeric(tag_i: str, tag_j: str):
    return named_matrix_entry(lamb_sol, tag_i, tag_j)


V_raw = []
partial_sum = lamb_numeric("x_star", "x_0") * f.interp_ineq("x_star", "x_0")
V_raw.append(partial_sum)

for step in range(N_int):
    partial_sum += lamb_numeric(f"x_{step}", f"x_{step + 1}") * f.interp_ineq(
        f"x_{step}", f"x_{step + 1}"
    )
    partial_sum += lamb_numeric("x_star", f"x_{step + 1}") * f.interp_ineq(
        "x_star", f"x_{step + 1}"
    )
    V_raw.append(partial_sum)

print("V_raw contains:", [f"V_{i}" for i in range(len(V_raw))])

V_raw contains: ['V_0', 'V_1', 'V_2', 'V_3', 'V_4', 'V_5']


## Step 2. Check for admissibility

### Sufficiency is immediate because $\mathcal I_N$ contains the adjacent interpolation constraints and all star interpolation constraints kept by the sparse certificate. We next inspect the ranks of the quadratic parts and the function-value support of each raw partial sum.

In [11]:
pm = pf.ExpressionManager(ctx_prf, resolve_parameters={"L": L_value, "R": R_value})
scalar_labels = [str(s) for s in ctx_prf.basis_scalars()]

for k, V in enumerate(V_raw):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-5)
    func_coords = V_eval.func_coords.astype(float)
    support = [scalar_labels[i] for i, val in enumerate(func_coords) if abs(val) > 1e-7]
    print(f"V_{k}: rank={rank}, nonzero function coordinates={support}")

print("\nFunction-value vector at last iteration:")
pf.pprint_labeled_vector(
    pm.eval_scalar(V_raw[N_int]).func_coords.astype(float),
    scalar_labels,
)

V_0: rank=2, nonzero function coordinates=['f(x_star)', 'f(x_0)']
V_1: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_1)']
V_2: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_2)']
V_3: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_3)']
V_4: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_4)']
V_5: rank=2, nonzero function coordinates=['f(x_star)', 'f(x_5)']

Function-value vector at last iteration:


<IPython.core.display.Math object>

## Step 3. Build a set of special candidate vectors

In [12]:
ctx_prf.set_as_current()
x = [ctx_prf[f"x_{i}"] for i in range(N_int + 1)]
z = [x[0]] + [ctx_prf[f"z_{i}"] for i in range(1, N_int + 1)]
x_0 = x[0]
x_star = ctx_prf["x_star"]

lyap_basis_candidate = list(ctx_prf.basis_vectors())
lyap_basis_candidate += x[1:] + z[1:]

base_count = len(lyap_basis_candidate)
for i, j in itertools.combinations(range(base_count), 2):
    diff = lyap_basis_candidate[i] - lyap_basis_candidate[j]
    lyap_basis_candidate.append(diff)

print(lyap_basis_candidate)

[x_0, x_star, grad_f(x_0), grad_f(x_1), grad_f(x_2), grad_f(x_3), grad_f(x_4), grad_f(x_5), x_1, x_2, x_3, x_4, x_5, z_1, z_2, z_3, z_4, z_5, x_0-x_star, x_0-grad_f(x_0), x_0-grad_f(x_1), x_0-grad_f(x_2), x_0-grad_f(x_3), x_0-grad_f(x_4), x_0-grad_f(x_5), x_0-(x_1), x_0-(x_2), x_0-(x_3), x_0-(x_4), x_0-(x_5), x_0-(z_1), x_0-(z_2), x_0-(z_3), x_0-(z_4), x_0-(z_5), x_star-grad_f(x_0), x_star-grad_f(x_1), x_star-grad_f(x_2), x_star-grad_f(x_3), x_star-grad_f(x_4), x_star-grad_f(x_5), x_star-(x_1), x_star-(x_2), x_star-(x_3), x_star-(x_4), x_star-(x_5), x_star-(z_1), x_star-(z_2), x_star-(z_3), x_star-(z_4), x_star-(z_5), grad_f(x_0)-grad_f(x_1), grad_f(x_0)-grad_f(x_2), grad_f(x_0)-grad_f(x_3), grad_f(x_0)-grad_f(x_4), grad_f(x_0)-grad_f(x_5), grad_f(x_0)-(x_1), grad_f(x_0)-(x_2), grad_f(x_0)-(x_3), grad_f(x_0)-(x_4), grad_f(x_0)-(x_5), grad_f(x_0)-(z_1), grad_f(x_0)-(z_2), grad_f(x_0)-(z_3), grad_f(x_0)-(z_4), grad_f(x_0)-(z_5), grad_f(x_1)-grad_f(x_2), grad_f(x_1)-grad_f(x_3), grad_f(x_

## Step 4. Find a basis of $\mathbf V_k$ within the candidate vectors

### We observe that all $V_k$ ($k=1,\dots,N-1$) contains a term $-\tau\|x_0-x_\star\|^2$, where $\tau=\tau_N$ is the value of the worst-case performance metric.

In [13]:
for k, V in enumerate(V_raw):
    print(
        f"V_{k}:",
        lu.vectors_in_column_space(
            V,
            lyap_basis_candidate,
            ctx_prf,
            resolve_parameters=pm.resolve_parameters,
            rtol=1e-5,
            atol=1e-5,
        ),
    )

V_0: [grad_f(x_0), x_0-x_star, x_0-(x_1), x_0-(z_1), x_star-(x_1), x_star-(z_1), x_1-(z_1)]
V_1: [grad_f(x_0), grad_f(x_1), x_0-x_star, x_0-(x_1), x_0-(x_2), x_0-(z_1), x_0-(z_2), x_star-(x_1), x_star-(x_2), x_star-(z_1), x_star-(z_2), grad_f(x_0)-grad_f(x_1), x_1-(x_2), x_1-(z_1), x_1-(z_2), x_2-(z_1), x_2-(z_2), z_1-(z_2)]
V_2: [grad_f(x_2), x_0-x_star, x_0-(z_2), x_0-(z_3), x_star-(z_2), x_star-(z_3), z_2-(z_3)]
V_3: [grad_f(x_3), x_0-x_star, x_0-(z_3), x_0-(z_4), x_star-(z_3), x_star-(z_4), z_3-(z_4)]
V_4: [grad_f(x_4), x_0-x_star, x_0-(z_4), x_0-(z_5), x_star-(z_4), x_star-(z_5), z_4-(z_5)]
V_5: [x_0-x_star]


In [14]:
for k, V in enumerate(V_raw):
    if k == 0 or k == len(V_raw) - 1:
        continue
    aligned_special_vectors = lu.vectors_in_column_space(
        V,
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        atol=1e-5,
    )
    best_vectors, best_coefficients = lu.find_basis_with_sparsest_coefficients(
        V,
        aligned_special_vectors,
        pep_context=ctx_prf,
        resolve_parameters=pm.resolve_parameters,
    )
    labels = [str(v) for v in best_vectors]
    print(f"V_{k}:")
    pf.pprint_labeled_matrix(best_coefficients, labels, labels)

V_1:


<IPython.core.display.Math object>

V_2:


<IPython.core.display.Math object>

V_3:


<IPython.core.display.Math object>

V_4:


<IPython.core.display.Math object>

### Consider the translated Lyapunov function $\widetilde V_k = V_k + \tau\|x_0-x_\star\|^2$.
### Then, we re-run the Steps 2-4 to verify that this is a desirable translation.

In [15]:
lyap = [V + tau_sol * (x_0 - x_star) ** 2 for V in V_raw]

for k, V in enumerate(lyap):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-5)
    print(f"tilde V_{k}: rank={rank}")

tilde V_0: rank=2
tilde V_1: rank=2
tilde V_2: rank=2
tilde V_3: rank=2
tilde V_4: rank=2
tilde V_5: rank=1


In [16]:
for idx in range(len(lyap)):
    aligned_special_vectors = lu.vectors_in_column_space(
        lyap[idx],
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        atol=1e-5,
    )
    print(f"tilde V_{idx}:")
    pf.pprint_labeled_vector(
        pm.eval_scalar(lyap[idx]).func_coords.astype(float),
        scalar_labels,
    )
    if not aligned_special_vectors:
        print("No candidate vector detected")
        continue
    else:
        best_vectors, best_coefficients = lu.find_basis_with_sparsest_coefficients(
            lyap[idx],
            aligned_special_vectors,
            pep_context=ctx_prf,
            resolve_parameters=pm.resolve_parameters,
        )
        labels = [str(v) for v in best_vectors]
        pf.pprint_labeled_matrix(best_coefficients, labels, labels)

tilde V_0:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

tilde V_1:


<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

tilde V_5:


<IPython.core.display.Math object>

No candidate vector detected


### We can hypothesize the form $\widetilde V_k = a_k (f(x_k) - f(x_\star)) + b_k \|\nabla f(x_k)\|^2 + c \|z_{k+1}-x_\star\|^2$ where $c$ is a constant. 
### The terminal $\widetilde V_N$ has to be handled separately because we have not found its basis vectors, which is probably due to the last step modification of OGM.

---

## Step 5. Analytic proof

### We now introduce only the analytic multipliers. For $i=0,\ldots,N-1$,
$$
    \lambda_{i,i+1}=\frac{2\theta_i^2}{\theta_N^2} \,\, (i=0,\ldots,N-1), \qquad 
    \lambda_{\star,i} = \begin{cases}
    \lambda_{i,i+1}-\lambda_{i-1,i} = 2\theta_i/\theta_N^2 , & 0 \le i\le N-1,\\
    1-\lambda_{N-1,N} = 1/\theta_N , & i=N.
\end{cases}
$$
### where we interpret $\lambda_{-1,0} = 0$.

### Verifying the analytic formulas:

In [17]:
def lambda_adj_formula(i: int, N: int):
    if 0 <= i <= N - 1:
        return 2 * theta(i, N) ** 2 / theta(N, N) ** 2
    return 0


def lambda_star_formula(i: int, N: int):
    if i == 0:
        return lambda_adj_formula(0, N)
    if 1 <= i <= N - 1:
        return lambda_adj_formula(i, N) - lambda_adj_formula(i - 1, N)
    if i == N:
        return 1 - lambda_adj_formula(N - 1, N)
    return 0


def lambda_formula(tag_i: str, tag_j: str, N: int):
    i = tag_to_index(tag_i, N)
    j = tag_to_index(tag_j, N)
    if tag_i == "x_star" and 0 <= j <= N:
        return lambda_star_formula(j, N)
    if 0 <= i <= N - 1 and j == i + 1:
        return lambda_adj_formula(i, N)
    return 0


lamb_cand = pf.pprint_labeled_matrix(
    lambda tag_i, tag_j: lambda_formula(tag_i, tag_j, N_int),
    lamb_sol.row_names,
    lamb_sol.col_names,
    return_matrix=True,
)
print(
    "closed-form lambdas match numerical certificate?",
    np.allclose(np.asarray(lamb_cand, dtype=float), lamb_sol.matrix, atol=1e-4),
)

tau_formula = L_value / (2 * theta(N_int, N_int) ** 2)
print(
    "closed-form tau matches numerical certificate?", np.allclose(tau_formula, tau_sol)
)
print("tau formula:", tau_formula)
print("tau numerical:", tau_sol)

<IPython.core.display.Math object>

closed-form lambdas match numerical certificate? True
closed-form tau matches numerical certificate? True
tau formula: 0.0185881366636511
tau numerical: 0.018588155914589344


### Symbolic verification helpers

In [18]:
def simplify_sympy_matrix(array):
    return sp.Matrix(array).applyfunc(
        lambda entry: sp.factor(sp.together(sp.nsimplify(entry)))
    )


def reduce_by_quadratic_recurrence(expr, eliminated_symbol, recurrence_expr):
    """Reduce a rational expression modulo a quadratic recurrence."""
    expr = sp.factor(sp.together(sp.simplify(expr)))
    numerator, denominator = sp.fraction(expr)
    if numerator == 0:
        return sp.S(0)
    try:
        numerator_reduced = sp.rem(
            sp.Poly(sp.expand(numerator), eliminated_symbol),
            sp.Poly(recurrence_expr, eliminated_symbol),
        ).as_expr()
        return sp.factor(sp.together(sp.simplify(numerator_reduced / denominator)))
    except sp.PolynomialError:
        return expr


def symbolic_parts(expr, ctx, resolve_parameters):
    pm_check = pf.ExpressionManager(ctx, resolve_parameters=resolve_parameters)
    matrix_part = simplify_sympy_matrix(pm_check.eval_scalar(expr).inner_prod_coords)
    func_part = simplify_sympy_matrix(pm_check.eval_scalar(expr).func_coords)
    return matrix_part, func_part


def display_symbolic_residual(name, matrix_part, func_part):
    print(name)
    print("quadratic residual:")
    display(matrix_part)
    print("function-value residual:")
    display(func_part)
    print(
        "identity verified?",
        matrix_part == sp.zeros(*matrix_part.shape)
        and func_part == sp.zeros(*func_part.shape),
    )

### We find the Lyapunov coefficients for $0\le k\le N-2$ in 
$$
\widetilde V_k
=a_k\bigl(f(x_k)-f(x_\star)\bigr)
+b_k\|\nabla f(x_k)\|^2
+c\|z_{k+1}-x_\star\|^2,
$$
### by solving 
$$
0=\widetilde V_k-\widetilde V_{k+1}
-\lambda_{k,k+1} I_{k,k+1}
-\lambda_{\star,k+1}I_{\star,k+1}
$$
### Here $z_{k+1}$ is the genuine OGM auxiliary vector after the $k$th $z$-update, so no index shift is being introduced.
### and reducing expressions using $\theta_{k+1}^2-\theta_{k+1}=\theta_k^2$.

In [19]:
ctx_ogm_interior = pf.PEPContext("ogm_interior_coefficient_proof").set_as_current()

x_k_abs = pf.Vector(is_basis=True, tags=["x_k"])
x_star_abs = pf.Vector(is_basis=True, tags=["x_{star}"])
z_kp1_abs = pf.Vector(is_basis=True, tags=["z_{k+1}"])
g_k_abs = pf.Vector(is_basis=True, tags=["g_k"])
g_kp1_abs = pf.Vector(is_basis=True, tags=["g_{k+1}"])

f_k_abs = pf.Scalar(is_basis=True, tags=["f_k"])
f_kp1_abs = pf.Scalar(is_basis=True, tags=["f_{k+1}"])
f_star_abs = pf.Scalar(is_basis=True, tags=["f_{star}"])

theta_k_par = pf.Parameter("theta_k")
theta_kp1_par = pf.Parameter("theta_{k+1}")
lambda_cons_par = pf.Parameter("lambda_{k,k+1}")
lambda_star_par = pf.Parameter("lambda_{star,k+1}")

y_k_abs = x_k_abs - sp.S(1) / L * g_k_abs
x_kp1_abs = (1 - sp.S(1) / theta_kp1_par) * y_k_abs + sp.S(
    1
) / theta_kp1_par * z_kp1_abs
z_kp2_abs = z_kp1_abs - sp.S(2) * theta_kp1_par / L * g_kp1_abs

a_k_par = pf.Parameter("a_k")
a_kp1_par = pf.Parameter("a_{k+1}")
b_k_par = pf.Parameter("b_k")
b_kp1_par = pf.Parameter("b_{k+1}")
c_par = pf.Parameter("c")

V_k_abs = (
    a_k_par * (f_k_abs - f_star_abs)
    + b_k_par * g_k_abs**2
    + c_par * (z_kp1_abs - x_star_abs) ** 2
)
V_kp1_abs = (
    a_kp1_par * (f_kp1_abs - f_star_abs)
    + b_kp1_par * g_kp1_abs**2
    + c_par * (z_kp2_abs - x_star_abs) ** 2
)

I_cons_abs = (
    (f_k_abs - f_kp1_abs)
    + g_kp1_abs * (x_kp1_abs - x_k_abs)
    - sp.S(1) / (2 * L) * (g_k_abs - g_kp1_abs) ** 2
)
I_star_abs = (
    (f_star_abs - f_kp1_abs)
    + g_kp1_abs * (x_kp1_abs - x_star_abs)
    - sp.S(1) / (2 * L) * g_kp1_abs**2
)

interior_diff = (
    V_k_abs - V_kp1_abs - lambda_cons_par * I_cons_abs - lambda_star_par * I_star_abs
)

In [20]:
L_sym = sp.Symbol("L")
theta_k_sym = sp.Symbol(r"\theta_k")
theta_kp1_sym = sp.Symbol(r"\theta_{k+1}")
theta_N_sym = sp.Symbol(r"\theta_N")
lambda_cons_sym = sp.Symbol(r"\lambda_{k,k+1}")
lambda_star_sym = sp.Symbol(r"\lambda_{star,k+1}")

a_k_sym, a_kp1_sym, b_k_sym, b_kp1_sym, c_sym = sp.symbols(r"a_k a_{k+1} b_k b_{k+1} c")

interior_resolve = {
    "L": L_sym,
    "theta_k": theta_k_sym,
    "theta_{k+1}": theta_kp1_sym,
    "lambda_{k,k+1}": lambda_cons_sym,
    "lambda_{star,k+1}": lambda_star_sym,
    "a_k": a_k_sym,
    "a_{k+1}": a_kp1_sym,
    "b_k": b_k_sym,
    "b_{k+1}": b_kp1_sym,
    "c": c_sym,
}

interior_matrix, interior_func = symbolic_parts(
    interior_diff, ctx_ogm_interior, interior_resolve
)
row_index = [str(v) for v in ctx_ogm_interior.basis_vectors()]
func_index = [str(s) for s in ctx_ogm_interior.basis_scalars()]

pf.pprint_labeled_matrix(
    np.array(interior_matrix.tolist(), dtype=object),
    row_index,
    row_index,
    precision=None,
)
pf.pprint_labeled_vector(
    np.array(interior_func, dtype=object).reshape(-1),
    func_index,
    precision=None,
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [21]:
interior_lambda_subs = {
    lambda_cons_sym: 2 * theta_k_sym**2 / theta_N_sym**2,
    lambda_star_sym: 2 * theta_kp1_sym / theta_N_sym**2,
}
interior_recurrence = theta_k_sym**2 - theta_kp1_sym**2 + theta_kp1_sym


def reduce_interior(expr):
    return reduce_by_quadratic_recurrence(
        expr.subs(interior_lambda_subs),
        theta_k_sym,
        interior_recurrence,
    )


coefficient_unknowns = (
    a_k_sym,
    a_kp1_sym,
    b_k_sym,
    b_kp1_sym,
    c_sym,
)
interior_equations = [
    reduce_interior(entry)
    for entry in list(interior_matrix) + list(interior_func)
    if any(entry.has(unknown) for unknown in coefficient_unknowns)
]
interior_equations = [eq for eq in interior_equations if eq != 0]

coefficient_matrix, rhs = sp.linear_eq_to_matrix(
    interior_equations, coefficient_unknowns
)
nullspace = coefficient_matrix.nullspace()
print("coefficient matrix rank:", coefficient_matrix.rank())
print("coefficient nullity:", len(nullspace))

solution_tuple = next(
    iter(sp.linsolve((coefficient_matrix, rhs), coefficient_unknowns))
)
solved_solution = {
    unknown: sp.factor(sp.together(sp.simplify(expression)))
    for unknown, expression in zip(coefficient_unknowns, solution_tuple)
}

print("Solved coefficient tuple:")
display(
    Math(
        r"\begin{aligned}"
        + r"\\".join(
            sp.latex(sp.Eq(unknown, solved_solution[unknown]))
            for unknown in coefficient_unknowns
        )
        + r"\end{aligned}"
    )
)

coefficient matrix rank: 5
coefficient nullity: 0
Solved coefficient tuple:


<IPython.core.display.Math object>

In [22]:
closed_form_solution = {
    a_k_sym: 2 * theta_k_sym**2 / theta_N_sym**2,
    a_kp1_sym: 2 * theta_kp1_sym**2 / theta_N_sym**2,
    b_k_sym: -(theta_k_sym**2) / (L_sym * theta_N_sym**2),
    b_kp1_sym: -(theta_kp1_sym**2) / (L_sym * theta_N_sym**2),
    c_sym: L_sym / (2 * theta_N_sym**2),
}

print("Closed-form coefficients, written in the original theta_k indexing:")
display(
    Math(
        r"\begin{aligned}"
        + r"\\".join(
            sp.latex(sp.Eq(unknown, closed_form_solution[unknown]))
            for unknown in coefficient_unknowns
        )
        + r"\end{aligned}"
    )
)

print(
    "Solved tuple equals the closed form under the recurrence?",
    all(
        reduce_by_quadratic_recurrence(
            solved_solution[unknown] - closed_form_solution[unknown],
            theta_k_sym,
            interior_recurrence,
        )
        == 0
        for unknown in coefficient_unknowns
    ),
)

Closed-form coefficients, written in the original theta_k indexing:


<IPython.core.display.Math object>

Solved tuple equals the closed form under the recurrence? True


In [23]:
verification_subs = {**interior_lambda_subs, **closed_form_solution}


def verify_interior_entry(entry):
    return reduce_by_quadratic_recurrence(
        entry.subs(verification_subs),
        theta_k_sym,
        interior_recurrence,
    )


interior_matrix_verified = interior_matrix.applyfunc(verify_interior_entry)
interior_func_verified = interior_func.applyfunc(verify_interior_entry)

display_symbolic_residual(
    "nonterminal transition residual",
    interior_matrix_verified,
    interior_func_verified,
)

nonterminal transition residual
quadratic residual:


Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0]])

identity verified? True


### We handle the final step separately, because it has different $\theta$-formula and $\theta_N^2-\theta_N=2\theta_{N-1}^2$, and $\lambda_{\star,N}=1-2\theta_{N-1}^2/\theta_N^2=1/\theta_N$ is also different from the rest of the indices.
### We start from the identity $\widehat V_N = \widetilde V_{N-1} - \lambda_{N-1,N}I_{N-1,N} - \lambda_{\star,N}I_{\star,N}$ and discover the analytic formula for $\widetilde V_N$.

In [24]:
ctx_ogm_last = pf.PEPContext("ogm_last_endpoint_proof").set_as_current()

x_Nm1_abs = pf.Vector(is_basis=True, tags=["x_{N-1}"])
x_star_last = pf.Vector(is_basis=True, tags=["x_{star}"])
z_N_abs = pf.Vector(is_basis=True, tags=["z_N"])
g_Nm1_abs = pf.Vector(is_basis=True, tags=["g_{N-1}"])
g_N_abs = pf.Vector(is_basis=True, tags=["g_N"])

f_Nm1_abs = pf.Scalar(is_basis=True, tags=["f_{N-1}"])
f_N_abs = pf.Scalar(is_basis=True, tags=["f_N"])
f_star_last = pf.Scalar(is_basis=True, tags=["f_{star}"])

theta_Nm1_par = pf.Parameter("theta_{N-1}")
theta_N_par = pf.Parameter("theta_N")
lambda_Nm1_N_par = pf.Parameter("lambda_{N-1,N}")
lambda_star_N_par = pf.Parameter("lambda_{star,N}")

y_Nm1_abs = x_Nm1_abs - sp.S(1) / L * g_Nm1_abs
x_N_abs = (1 - sp.S(1) / theta_N_par) * y_Nm1_abs + sp.S(1) / theta_N_par * z_N_abs

V_Nm1_abs = (
    2 * theta_Nm1_par**2 / theta_N_par**2 * (f_Nm1_abs - f_star_last)
    - theta_Nm1_par**2 / (L * theta_N_par**2) * g_Nm1_abs**2
    + L / (2 * theta_N_par**2) * (z_N_abs - x_star_last) ** 2
)

I_Nm1_N_abs = (
    (f_Nm1_abs - f_N_abs)
    + g_N_abs * (x_N_abs - x_Nm1_abs)
    - sp.S(1) / (2 * L) * (g_Nm1_abs - g_N_abs) ** 2
)
I_star_N_abs = (
    (f_star_last - f_N_abs)
    + g_N_abs * (x_N_abs - x_star_last)
    - sp.S(1) / (2 * L) * g_N_abs**2
)

V_N_from_step = (
    V_Nm1_abs - lambda_Nm1_N_par * I_Nm1_N_abs - lambda_star_N_par * I_star_N_abs
)

In [25]:
theta_Nm1_sym = sp.Symbol(r"\theta_{N-1}")
theta_N_last_sym = sp.Symbol(r"\theta_N")
lambda_Nm1_N_sym = sp.Symbol(r"\lambda_{N-1,N}")
lambda_star_N_sym = sp.Symbol(r"\lambda_{\star,N}")

last_resolve = {
    "L": L_sym,
    "theta_{N-1}": theta_Nm1_sym,
    "theta_N": theta_N_last_sym,
    "lambda_{N-1,N}": lambda_Nm1_N_sym,
    "lambda_{star,N}": lambda_star_N_sym,
}

last_matrix, last_func = symbolic_parts(V_N_from_step, ctx_ogm_last, last_resolve)
last_lambda_subs = {
    lambda_Nm1_N_sym: 2 * theta_Nm1_sym**2 / theta_N_last_sym**2,
    lambda_star_N_sym: 1 - 2 * theta_Nm1_sym**2 / theta_N_last_sym**2,
}
last_recurrence = 2 * theta_Nm1_sym**2 - theta_N_last_sym**2 + theta_N_last_sym


def reduce_last_entry(entry):
    return reduce_by_quadratic_recurrence(
        entry.subs(last_lambda_subs),
        theta_Nm1_sym,
        last_recurrence,
    )


last_matrix_discovered = last_matrix.applyfunc(reduce_last_entry)
last_func_discovered = last_func.applyfunc(reduce_last_entry)
last_row_index = [str(v) for v in ctx_ogm_last.basis_vectors()]
last_func_index = [str(s) for s in ctx_ogm_last.basis_scalars()]

print("function-value part discovered from the terminal identity:")
pf.pprint_labeled_vector(
    np.array(last_func_discovered, dtype=object).reshape(-1),
    last_func_index,
    precision=None,
)
print("quadratic part discovered from the terminal identity:")
pf.pprint_labeled_matrix(
    np.array(last_matrix_discovered.tolist(), dtype=object),
    last_row_index,
    last_row_index,
    precision=None,
)
print("terminal quadratic rank:", last_matrix_discovered.rank())

terminal_vector_abs = z_N_abs - x_star_last - theta_N_par / L * g_N_abs
terminal_square = L / (2 * theta_N_par**2) * terminal_vector_abs**2
terminal_target = f_N_abs - f_star_last + terminal_square
terminal_residual = V_N_from_step - terminal_target
terminal_residual_matrix, terminal_residual_func = symbolic_parts(
    terminal_residual, ctx_ogm_last, last_resolve
)
terminal_residual_matrix = terminal_residual_matrix.applyfunc(reduce_last_entry)
terminal_residual_func = terminal_residual_func.applyfunc(reduce_last_entry)

print("rank-one vector read from the displayed matrix:")
display(Math(r"v_N=z_N-x_\star-\frac{\theta_N}{L}\nabla f(x_N)"))
display_symbolic_residual(
    "terminal closed form residual",
    terminal_residual_matrix,
    terminal_residual_func,
)

function-value part discovered from the terminal identity:


<IPython.core.display.Math object>

quadratic part discovered from the terminal identity:


<IPython.core.display.Math object>

terminal quadratic rank: 1
rank-one vector read from the displayed matrix:


...

terminal closed form residual
quadratic residual:


Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0]])

identity verified? True


### The code above discovers
$$
\widetilde V_N
=f(x_N)-f(x_\star)+\frac{L}{2\theta_N^2}
\left\|z_N-x_\star-\frac{\theta_N}{L}\nabla f(x_N)\right\|^2 .
$$